# 规模因子 (SMB) 完整教程：从市值分组到多空组合检验

## 📚 教学目标
1. 理解**规模因子**的定义：按市值分组，小市值组 − 大市值组 = SMB
2. 掌握 **A 股**中市值分组的具体操作（总市值、对数处理）
3. 在**微型数据集**（10 只股票 × 1 月）上手算分组和 Spread
4. 扩展到 **300 只股票 × 60 月**，完成 T 检验和单调性检验
5. 理解 **A 股小市值效应**的历史表现和风险

**参考书**：《因子投资：方法与实践》（石川）第 3.3 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 什么是规模因子？为什么小市值股票收益更高？

### 🎯 一个直觉问题

假设你面前有两家公司：A 公司市值 50 亿，B 公司市值 5000 亿。其他条件相同，你更愿意买哪家？

学术研究发现，小市值股票长期来看收益更高。**规模因子 (SMB, Small-Minus-Big)** 就是检验这个现象：做多小市值股票、做大市值股票，能否获得显著的超额收益？

### 📖 书中的定义

> 规模效应（Size）的含义简单清晰，即小市值股票有着比大市值股票更好的表现，而反映这一现象的规模因子则因为著名的 Fama and French（1993）三因子模型而家喻户晓。

> Banz（1981）基于纽约证券交易所（NYSE）长达 40 年的数据发现，按照市值将股票分为 5 组后，市值最小的一组股票月均收益率比其他股票高 0.4%，差异非常显著。

### 📐 规模因子的理论基础

| 理论来源 | 核心逻辑 |
|---------|----------|
| **风险补偿说** | 小市值公司财务困境风险更大 → 需要更高回报补偿 |
| **非流动性溢价** | 小市值股票流动性差 → 要求更高收益 |
| **投资者行为** | 机构偏好大市值 → 小盘股缺乏投资者基础 → 估值偏低 |
| **遗漏变量偏误** | 定价模型设定有误 → 规模成为代理变量 |

### 📐 SMB 因子的定义

$$\text{SMB} = R_{\text{Small}} - R_{\text{Big}}$$

其中：
- $R_{\text{Small}}$ = 小市值组（如市值最小的 30%）的组合收益率
- $R_{\text{Big}}$ = 大市值组（如市值最大的 30%）的组合收益率
- SMB > 0 表示小市值股票收益高于大市值股票

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据集：10 只股票的手算演示

### 🎯 场景

假设市场上只有 **10 只股票**，我们用总市值作为排序变量，检验小市值股票是否获得更高收益。

### 📐 数据生成逻辑

$$r_i = \bar{r} + \gamma \cdot (\overline{\text{Size}} - \text{Size}_i) + \varepsilon_i$$

其中：
- $\bar{r}$ = 基础收益率（所有股票共享）
- $\gamma$ = 规模效应系数（> 0 表示小市值带来高收益）
- $\varepsilon_i$ = 个股噪声

In [ ]:
# ========== 构造 10 只股票的微型数据 ==========
np.random.seed(42)

stocks = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']

# 总市值（亿元）：从大到小
market_cap = np.array([5000, 3000, 1500, 800, 400, 200, 100, 50, 20, 5])

# 数据生成参数
base_return = 1.0    # 月基础收益率 1%
gamma = 0.00015      # 规模效应：每 1 亿市值差异 → 0.00015% 收益差异
noise = np.random.normal(0, 2.0, 10)

# 收益率 = 基础 + 规模效应 + 噪声（小市值收益更高）
log_cap = np.log(market_cap)
returns = base_return + gamma * (log_cap.mean() - log_cap) * 100 + noise

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': stocks,
    'MarketCap (亿)': market_cap,
    'Return (%)': np.round(returns, 2)
})

print("📋 10 只股票数据集：")
print(df_micro.to_string(index=False))
print(f"\n💡 规模效应系数 γ = {gamma}（小市值股票收益更高）")
print(f"   基础收益率 = {base_return}%，噪声标准差 = 2.0%")

### 📐 步骤 1: 按市值排序分组

将 10 只股票按市值从大到小排序，分为 2 组：
- **Big 组（Q1）**：市值最大的 5 只 → 做空
- **Small 组（Q2）**：市值最小的 5 只 → 做多

$$\text{SMB} = \bar{R}_{\text{Small}} - \bar{R}_{\text{Big}}$$

In [ ]:
# ========== 步骤 1: 按市值排序分组 ==========
print("📊 步骤 1: 按市值排序分组")
print("─" * 55)

df_sorted = df_micro.sort_values('MarketCap (亿)', ascending=False).reset_index(drop=True)
df_sorted['Group'] = ['Big'] * 5 + ['Small'] * 5

print("\n  Big 市值组（做空）:")
for _, row in df_sorted[df_sorted['Group'] == 'Big'].iterrows():
    print(f"    {row['Stock']}: 市值 = {row['MarketCap (亿)']:>6}亿,  收益 = {row['Return (%)']:>6.2f}%")

print("\n  Small 市值组（做多）:")
for _, row in df_sorted[df_sorted['Group'] == 'Small'].iterrows():
    print(f"    {row['Stock']}: 市值 = {row['MarketCap (亿)']:>6}亿,  收益 = {row['Return (%)']:>6.2f}%")

In [ ]:
# ========== 步骤 2: 计算各组平均收益率和 Spread ==========
print("📊 步骤 2: 计算各组平均收益率和 Spread")
print("─" * 55)

big_mean = df_sorted[df_sorted['Group'] == 'Big']['Return (%)'].mean()
small_mean = df_sorted[df_sorted['Group'] == 'Small']['Return (%)'].mean()
spread = small_mean - big_mean

print(f"\n  Big 组平均收益率    = {big_mean:.2f}%")
print(f"  Small 组平均收益率  = {small_mean:.2f}%")
print(f"\n  📐 SMB = Small - Big = {small_mean:.2f}% - {big_mean:.2f}% = {spread:.2f}%")
print(f"\n  💡 解读：小市值股票比大市值股票{'多' if spread > 0 else '少'}赚 {abs(spread):.2f}%")

---

## 3. 扩展到完整模拟：300 只股票 × 60 月

### 📐 数据生成过程 (DGP)

每月生成 300 只股票的截面数据：

$$r_{i,t} = \bar{r}_t + \gamma \cdot (\overline{\log(\text{Cap})} - \log(\text{Cap}_{i,t})) + \varepsilon_{i,t}$$

其中：
- $\bar{r}_t$ = 每月基础收益率（从 $N(1.0, 0.5)$ 抽样）
- $\gamma$ = 规模效应强度
- $\varepsilon_{i,t} \sim N(0, 3)$ = 个股噪声

In [ ]:
# ========== 完整模拟参数 ==========
N_STOCKS = 300    # 每月 300 只股票
N_MONTHS = 60     # 60 个月
N_GROUPS = 5      # 分 5 组

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS} 组")

In [ ]:
# ========== 生成模拟数据 ==========
np.random.seed(42)

all_data = []
for t in range(N_MONTHS):
    # 市值：对数正态分布（模拟 A 股市值分布）
    log_cap = np.random.normal(loc=5.0, scale=1.5, size=N_STOCKS)
    market_cap = np.exp(log_cap)
    
    # 基础收益率（每月不同）
    base_return = np.random.normal(1.0, 0.5)
    
    # 规模效应：小市值收益更高
    gamma = 0.3
    size_effect = gamma * (log_cap.mean() - log_cap)
    
    # 噪声
    noise = np.random.normal(0, 3.0, N_STOCKS)
    
    # 收益率
    returns = base_return + size_effect + noise
    
    month_df = pd.DataFrame({
        'Month': t + 1,
        'Stock': [f'S{i:03d}' for i in range(N_STOCKS)],
        'MarketCap': market_cap,
        'LogCap': log_cap,
        'Return': returns
    })
    all_data.append(month_df)

df_all = pd.concat(all_data, ignore_index=True)

print(f"✅ 数据生成完成：{len(df_all)} 条记录")
print(f"   股票数: {df_all['Stock'].nunique()}")
print(f"   月份数: {df_all['Month'].nunique()}")
print(f"\n📊 市值分布统计：")
print(df_all['MarketCap'].describe().round(2))

### 📐 步骤 3: 每月按市值分组

每月末将股票按市值从低到高分成 5 组：
- Q1 = 市值最小（Small）
- Q5 = 市值最大（Big）

$$\text{SMB}_t = \bar{R}_{Q1,t} - \bar{R}_{Q5,t}$$

In [ ]:
# ========== 每月按市值分组 ==========
def assign_size_group(month_df):
    """按市值从低到高分 5 组"""
    month_df = month_df.copy()
    month_df['SizeGroup'] = pd.qcut(month_df['MarketCap'], N_GROUPS, 
                                     labels=['Q1(Small)', 'Q2', 'Q3', 'Q4', 'Q5(Big)'])
    return month_df

df_all = df_all.groupby('Month', group_keys=False).apply(assign_size_group)

# 计算每月各组平均收益率
monthly_group_returns = df_all.groupby(['Month', 'SizeGroup'])['Return'].mean().unstack()

# 计算每月 SMB = Q1 - Q5
monthly_spreads = monthly_group_returns['Q1(Small)'] - monthly_group_returns['Q5(Big)']

print("📊 每月各组平均收益率（前 5 个月）：")
print(monthly_group_returns.head().round(3))
print(f"\n📊 每月 SMB Spread（前 5 个月）：")
print(monthly_spreads.head().round(3))

### 📐 步骤 4: T 检验——SMB 是否显著大于零？

$$t = \frac{\overline{\text{SMB}}}{s_{\text{SMB}} / \sqrt{T}}$$

其中：
- $\overline{\text{SMB}}$ = SMB 时间序列均值
- $s_{\text{SMB}}$ = SMB 样本标准差
- $T$ = 月份数

In [ ]:
# ========== T 检验 ==========
spreads = monthly_spreads.values
spread_mean = spreads.mean()
spread_std = spreads.std(ddof=1)
spread_se = spread_std / np.sqrt(N_MONTHS)
t_stat = spread_mean / spread_se
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=N_MONTHS - 1))

print("📊 步骤 4: T 检验结果")
print("─" * 55)
print(f"  SMB 均值       = {spread_mean:.4f}%")
print(f"  SMB 标准差     = {spread_std:.4f}%")
print(f"  标准误         = {spread_se:.4f}%")
print(f"  T 统计量       = {t_stat:.4f}")
print(f"  P 值 (双尾)    = {p_value:.6f}")
print(f"\n  💡 解读：")
if abs(t_stat) > 2.6:
    print(f"  ✓ |t| = {abs(t_stat):.2f} > 2.6 → 在 1% 水平下显著")
elif abs(t_stat) > 2.0:
    print(f"  ✓ |t| = {abs(t_stat):.2f} > 2.0 → 在 5% 水平下显著")
else:
    print(f"  ✗ |t| = {abs(t_stat):.2f} < 2.0 → 不显著")

In [ ]:
# ========== 用 scipy 验证 ==========
t_scipy, p_scipy = stats.ttest_1samp(spreads, 0)

print("🔬 scipy.stats.ttest_1samp 验证:")
print(f"  手算 T 统计量 = {t_stat:.6f}")
print(f"  scipy T 统计量 = {t_scipy:.6f}")
print(f"  手算 P 值     = {p_value:.6f}")
print(f"  scipy P 值    = {p_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

### 📐 步骤 5: 经济意义——夏普比率

$$\text{SR}_{\text{monthly}} = \frac{\overline{\text{SMB}}}{s_{\text{SMB}}}$$

$$\text{SR}_{\text{annual}} = \text{SR}_{\text{monthly}} \times \sqrt{12}$$

In [ ]:
# ========== 夏普比率 ==========
sr_monthly = spread_mean / spread_std
sr_annual = sr_monthly * np.sqrt(12)

print("📊 步骤 5: 夏普比率")
print("─" * 55)
print(f"  月度夏普比率 = {sr_monthly:.4f}")
print(f"  年化夏普比率 = {sr_annual:.4f}")
print(f"\n  📐 验证: t = SR_monthly × √T = {sr_monthly:.4f} × √{N_MONTHS} = {sr_monthly * np.sqrt(N_MONTHS):.4f}")
print(f"\n  💡 解读：年化 SR > 0.5 表示因子有较好的风险调整后收益")

---

## 4. 单调性检验：各组收益率是否单调递减？

### 📐 Spearman 秩相关

检验市值分组（Q1 到 Q5）与平均收益率之间是否存在单调关系：

$$\rho_s = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$$

- $|\rho_s| > 0.8$：强单调性
- $|\rho_s| > 0.5$：中等单调性
- $|\rho_s| < 0.5$：弱单调性

In [ ]:
# ========== 单调性检验 ==========
avg_group_returns = monthly_group_returns.mean()
group_ranks = np.arange(1, N_GROUPS + 1)
group_ret_values = avg_group_returns.values

sp_corr, sp_p = stats.spearmanr(group_ranks, group_ret_values)

print("📊 单调性检验结果")
print("─" * 55)
print(f"  各组平均收益率：")
for i, (group, ret) in enumerate(avg_group_returns.items()):
    print(f"    {group}: {ret:.4f}%")
print(f"\n  Spearman 秩相关系数 = {sp_corr:.4f}")
print(f"  P 值 = {sp_p:.6f}")
print(f"\n  💡 解读：")
if abs(sp_corr) > 0.8:
    print(f"  ✓ |ρ| = {abs(sp_corr):.2f} > 0.8 → 强单调性，因子质量优秀")
elif abs(sp_corr) > 0.5:
    print(f"  ✓ |ρ| = {abs(sp_corr):.2f} > 0.5 → 中等单调性")
else:
    print(f"  ✗ |ρ| = {abs(sp_corr):.2f} < 0.5 → 弱单调性，因子不可靠")

---

## 5. 可视化：规模效应的全景图

In [ ]:
# ========== 可视化 T 检验 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: Spread 时间序列 ---
ax1 = axes[0]
months = range(1, N_MONTHS + 1)
ax1.bar(months, spreads, color=['#2ecc71' if s > 0 else '#e74c3c' for s in spreads],
        alpha=0.7, edgecolor='none')
ax1.axhline(y=spread_mean, color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {spread_mean:.2f}%')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('SMB Spread (%)', fontsize=11)
ax1.set_title('Monthly SMB Spread (Small - Big)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: Spread 分布 ---
ax2 = axes[1]
ax2.hist(spreads, bins=15, edgecolor='black', alpha=0.7, color='steelblue', density=True)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='$H_0$: $\mu$=0')
ax2.axvline(x=spread_mean, color='blue', linestyle='--', linewidth=2,
            label=f'Sample Mean = {spread_mean:.2f}%')
x_fit = np.linspace(spreads.min() - 1, spreads.max() + 1, 100)
ax2.plot(x_fit, stats.norm.pdf(x_fit, spread_mean, spread_std), 'b-', linewidth=2)
ax2.set_xlabel('SMB Spread (%)', fontsize=11)
ax2.set_ylabel('Probability Density', fontsize=11)
ax2.set_title('SMB Spread Distribution', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

# --- 图3: T 分布与 T 值位置 ---
ax3 = axes[2]
x_t = np.linspace(-5, 5, 1000)
t_dist = stats.t.pdf(x_t, df=N_MONTHS - 1)
ax3.plot(x_t, t_dist, 'b-', linewidth=2, label=f'T distribution (df={N_MONTHS-1})')
ax3.fill_between(x_t, t_dist, where=(x_t >= abs(t_stat)), color='red', alpha=0.4)
ax3.fill_between(x_t, t_dist, where=(x_t <= -abs(t_stat)), color='red', alpha=0.4,
                 label=f'P-value region = {p_value:.4f}')
ax3.axvline(x=t_stat, color='red', linestyle='--', linewidth=2,
            label=f't = {t_stat:.2f}')
ax3.set_xlabel('T Value', fontsize=11)
ax3.set_ylabel('Probability Density', fontsize=11)
ax3.set_title('T Distribution and T Value Position', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：每月 SMB Spread 的时间序列，绿色为正（小市值跑赢），红色为负")
print(f"  图2：SMB Spread 的分布直方图，叠加正态拟合曲线")
print(f"  图3：T 分布图，红色区域为拒绝域，t 值落在该区域表示显著")

In [ ]:
# ========== 可视化单调性 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: 各组收益柱状图 ---
ax1 = axes[0]
group_labels = avg_group_returns.index.tolist()
group_vals = avg_group_returns.values
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, N_GROUPS))
bars = ax1.bar(group_labels, group_vals, color=colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars, group_vals):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{v:.3f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax1.plot(group_labels, group_vals, 'ko--', linewidth=2, markersize=8, zorder=5)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Size Group (Q1=Smallest → Q5=Biggest)', fontsize=11)
ax1.set_ylabel('Average Monthly Return (%)', fontsize=11)
ax1.set_title(f'Size Monotonicity (ρ = {sp_corr:.3f})', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 图2: 累计收益率 ---
ax2 = axes[1]
cum_smb = np.cumsum(spreads)
ax2.plot(range(1, N_MONTHS + 1), cum_smb, 'b-', linewidth=2)
ax2.fill_between(range(1, N_MONTHS + 1), cum_smb, alpha=0.3, color='steelblue')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('Cumulative SMB Return (%)', fontsize=11)
ax2.set_title('Cumulative SMB Factor Return', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

# --- 图3: 多空两头累计收益 ---
ax3 = axes[2]
cum_small = np.cumsum(monthly_group_returns['Q1(Small)'].values)
cum_big = np.cumsum(monthly_group_returns['Q5(Big)'].values)
ax3.plot(range(1, N_MONTHS + 1), cum_small, 'g-', linewidth=2, label='Small (Q1)')
ax3.plot(range(1, N_MONTHS + 1), cum_big, 'r-', linewidth=2, label='Big (Q5)')
ax3.axhline(y=0, color='black', linewidth=0.8)
ax3.set_xlabel('Month', fontsize=11)
ax3.set_ylabel('Cumulative Return (%)', fontsize=11)
ax3.set_title('Long and Short Leg Cumulative Returns', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：各市值组的平均月收益率，理想情况下应单调递减（小市值收益最高）")
print(f"  图2：SMB 因子的累计收益率曲线，上升趋势表示小市值持续跑赢")
print(f"  图3：多空两头各自的累计收益，绿色=小市值（做多），红色=大市值（做空）")

---

## 6. 结果汇总报告

In [ ]:
# ========== 汇总报告 ==========
print("=" * 60)
print("📋 规模因子 (SMB) 实证分析报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   小市值股票是否比大市值股票获得更高收益？")

print(f"\n📊 数据概况:")
print(f"   股票数量: {N_STOCKS} 只/月")
print(f"   时间跨度: {N_MONTHS} 个月")
print(f"   分组方式: 按市值从低到高分 {N_GROUPS} 组")

print(f"\n🧮 统计检验:")
print(f"   SMB 月均收益率 = {spread_mean:.4f}%")
print(f"   T 统计量      = {t_stat:.4f}")
print(f"   P 值           = {p_value:.6f}")
print(f"   月度夏普比率   = {sr_monthly:.4f}")
print(f"   年化夏普比率   = {sr_annual:.4f}")
print(f"   Spearman ρ    = {sp_corr:.4f}")

print(f"\n🎯 结论:")
if t_stat > 2.0 and sp_corr > 0.5:
    print(f"  ✓ 规模效应显著：小市值股票收益显著高于大市值股票")
    print(f"  ✓ 单调性良好：收益率随市值增加而单调递减")
elif t_stat > 2.0:
    print(f"  ✓ 规模效应显著，但单调性不佳")
else:
    print(f"  ✗ 规模效应不显著")

print("\n" + "=" * 60)

---

## 📌 核心概念回顾

### 📌 规模因子 (SMB)
- **定义**: Small-Minus-Big，做多小市值、做空大市值的多空组合
- **公式**: $\text{SMB} = R_{\text{Small}} - R_{\text{Big}}$
- **含义**: 反映小市值股票相对大市值股票的超额收益
- **判断标准**: t > 2.0 表示显著，Spearman ρ > 0.8 表示强单调性

### 📌 Fama-French 三因子模型中的 SMB
- **来源**: Fama and French (1993)
- **构建**: 按市值中位数分两组，SMB = 小市值组均值 - 大市值组均值
- **作用**: 控制规模效应，检验其他因子的增量解释力

### 📌 A 股规模效应的特殊性
- **壳价值污染**: 微型股壳价值畸高，扭曲规模溢价
- **2017 年后**: 大市值白马股崛起，规模效应被稀释
- **处理方法**: 剔除市值最低 30% 股票（Liu et al. 2019）

### 🔗 完整流程
```
市值数据 → 每月排序分组 → 计算各组收益率 → SMB = Small - Big
    ↓           ↓              ↓              ↓
  清洗       5 组          等权/市值加权      T 检验 + 单调性
```

---

## ❌ 常见误区

### ❌ 误区 1: 规模效应在所有市场都成立
**✓ 正确理解**: 规模效应有显著的时变性和市场差异。A 股在 2017 年后规模效应大幅减弱，日本市场甚至出现反转。

### ❌ 误区 2: 小市值股票一定比大市值股票好
**✓ 正确理解**: 小市值股票收益高是统计平均结果，个股层面风险很大（流动性差、信息不对称、退市风险）。

### ❌ 误区 3: 等权和市值加权的结果应该一样
**✓ 正确理解**: 等权对小市值有额外暴露，通常收益更高但波动也更大。学术研究通常同时报告两种结果。

### ❌ 误区 4: 规模因子的收益是"免费午餐"
**✓ 正确理解**: 因子收益是对承担风险的补偿。小市值股票面临更高的财务困境风险、流动性风险和信息不对称。

### ❌ 误区 5: 只看收益率就够了
**✓ 正确理解**: 必须同时检验统计显著性（t 值）和经济显著性（夏普比率）。高收益但高波动的因子可能只是噪声。